# COMP532 Assignment 2 — Problem 1
## Deep Reinforcement Learning for LunarLander

**Algorithm:** Dueling Double DQN (a strict superset of the starter's vanilla DQN).

This is a self-contained notebook. Run cells top-to-bottom. On a CPU, 1000 episodes takes about 20 minutes; the default below is set to **300 episodes** for a quicker first run — change `N_EPISODES` to `1000` to fully reproduce the report.

**What this notebook contains**
1. Hyperparameters and seeding
2. The Dueling Q-network (PyTorch)
3. The replay buffer (NumPy ring buffer)
4. The agent (epsilon-greedy + Double-DQN target + Polyak soft updates)
5. The training loop (the requested `learn()` is filled in inside the agent)
6. Plots (rewards vs episodes, training loss vs steps)
7. Greedy evaluation + GIF recording for the report


## 1. Setup

Install dependencies if needed (uncomment), then import.

In [ ]:
# !pip install -q gymnasium 'gymnasium[box2d]' torch numpy matplotlib imageio Pillow

In [ ]:
import os, math, random, json, time
from collections import deque
from dataclasses import dataclass, asdict, field
from typing import List, Optional, Tuple

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import gymnasium as gym
import imageio.v2 as imageio

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch', torch.__version__, '| Gymnasium', gym.__version__, '| Device', DEVICE)

## 2. Environment helper

`LunarLander-v2` (OpenAI Gym) and `LunarLander-v3` (Gymnasium ≥ 1.0) are the same environment with different version tags. We try v3 first and fall back to v2.

In [ ]:
def make_env(**kwargs):
    for cid in ('LunarLander-v3', 'LunarLander-v2'):
        try:
            return gym.make(cid, **kwargs)
        except Exception:
            continue
    raise RuntimeError('Could not create LunarLander env (need gymnasium[box2d]).')

_env = make_env()
STATE_DIM  = _env.observation_space.shape[0]
ACTION_DIM = _env.action_space.n
print(f'state_dim={STATE_DIM}, action_dim={ACTION_DIM}')
_env.close()

## 3. Hyperparameters

All knobs in one place. Defaults are tuned for LunarLander; only `N_EPISODES` is
lowered from the report's 1000 so the notebook finishes quickly on first run.

In [ ]:
@dataclass
class Config:
    state_dim: int        = STATE_DIM
    action_dim: int       = ACTION_DIM
    hidden: int           = 128
    gamma: float          = 0.99           # discount factor
    lr: float             = 5e-4           # Adam learning rate
    batch_size: int       = 64
    buffer_capacity: int  = 100_000
    min_buffer_for_train: int = 1_000
    tau: float            = 1e-3           # Polyak soft-update rate
    update_every: int     = 4              # gradient step every k env steps
    epsilon_start: float  = 1.0
    epsilon_end: float    = 0.01
    epsilon_decay: float  = 0.995          # multiplicative per episode
    grad_clip_norm: float = 10.0
    seed: int             = 42

cfg = Config()

# Training budget. The full reproduction in the report uses 1000 episodes (~20 min CPU).
N_EPISODES = 300       # set to 1000 to fully reproduce the report's curves
MAX_STEPS  = 1000

# Reproducibility
random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed)

## 4. Dueling Q-network

Shared trunk $\to$ two heads:
- $V(s)\in\mathbb R$ (state value)
- $A(s, a)\in\mathbb R^{|A|}$ (advantage)

Recombined as $Q(s, a) = V(s) + (A(s, a) - \frac{1}{|A|}\sum_{a'} A(s, a'))$.
Mean subtraction (rather than max) keeps the gradient differentiable everywhere.

In [ ]:
class DuelingQNetwork(nn.Module):
    """Two-head MLP: shared trunk -> V(s) and A(s, a)."""

    def __init__(self, state_dim: int, action_dim: int, hidden: int = 128, seed: int = 42):
        super().__init__()
        torch.manual_seed(seed)
        self.action_dim = action_dim

        self.trunk = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden),
            nn.ReLU(inplace=True),
        )
        self.value_head = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(inplace=True), nn.Linear(hidden, 1)
        )
        self.advantage_head = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(inplace=True), nn.Linear(hidden, action_dim)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, state: torch.Tensor) -> torch.Tensor:
        z = self.trunk(state)
        v = self.value_head(z)                       # (B, 1)
        a = self.advantage_head(z)                   # (B, A)
        return v + (a - a.mean(dim=1, keepdim=True))  # (B, A)

## 5. Replay buffer

Pre-allocated NumPy ring buffer for $(s, a, r, s', \text{done})$ — about 5× faster than a Python `deque` of tuples because there are no per-step Python allocations.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity: int, state_dim: int, device: torch.device, seed: int = 0):
        self.capacity = int(capacity); self.device = device
        self.idx = 0; self.size = 0
        self.states      = np.zeros((capacity, state_dim), dtype=np.float32)
        self.actions     = np.zeros(capacity, dtype=np.int64)
        self.rewards     = np.zeros(capacity, dtype=np.float32)
        self.next_states = np.zeros((capacity, state_dim), dtype=np.float32)
        self.dones       = np.zeros(capacity, dtype=np.float32)
        np.random.seed(seed)

    def push(self, s, a, r, s2, d):
        i = self.idx
        self.states[i] = s; self.actions[i] = a; self.rewards[i] = r
        self.next_states[i] = s2; self.dones[i] = float(d)
        self.idx = (self.idx + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size: int):
        idxs = np.random.randint(0, self.size, size=batch_size)
        s  = torch.from_numpy(self.states[idxs]).to(self.device)
        a  = torch.from_numpy(self.actions[idxs]).to(self.device)
        r  = torch.from_numpy(self.rewards[idxs]).to(self.device)
        s2 = torch.from_numpy(self.next_states[idxs]).to(self.device)
        d  = torch.from_numpy(self.dones[idxs]).to(self.device)
        return s, a, r, s2, d

    def __len__(self): return self.size

## 6. Dueling Double DQN agent

This is the cell that fills in the starter notebook's TODOs. Specifically:
- Two networks for fixed Q-targets ✔
- Sample a batch of experiences ✔
- Compute Q targets with the **Double-DQN** rule (action chosen by online net, evaluated by target net) ✔
- Smooth-L1 (Huber) loss + gradient clipping + Polyak soft target update ✔


In [ ]:
class DuelingDoubleDQNAgent:
    """Dueling Double DQN agent. Implements the starter's missing learn() steps."""

    def __init__(self, cfg: Config, device: torch.device = DEVICE):
        self.cfg = cfg; self.device = device

        # TWO networks for Fixed Q-Targets
        self.qnetwork_local  = DuelingQNetwork(cfg.state_dim, cfg.action_dim, cfg.hidden, cfg.seed).to(device)
        self.qnetwork_target = DuelingQNetwork(cfg.state_dim, cfg.action_dim, cfg.hidden, cfg.seed).to(device)
        self.qnetwork_target.load_state_dict(self.qnetwork_local.state_dict())
        for p in self.qnetwork_target.parameters():
            p.requires_grad_(False)

        self.optimizer = optim.Adam(self.qnetwork_local.parameters(), lr=cfg.lr)
        self.replay = ReplayBuffer(cfg.buffer_capacity, cfg.state_dim, device, cfg.seed)
        self.epsilon = cfg.epsilon_start
        self.t_step = 0

    # ----- action selection ---------------------------------------------- #
    @torch.no_grad()
    def select_action(self, state: np.ndarray, greedy: bool = False) -> int:
        if (not greedy) and np.random.random() < self.epsilon:
            return int(np.random.randint(self.cfg.action_dim))
        s = torch.from_numpy(state).float().unsqueeze(0).to(self.device)
        return int(self.qnetwork_local(s).argmax(dim=1).item())

    # ----- env step + maybe learn ---------------------------------------- #
    def step(self, s, a, r, s2, d) -> Optional[float]:
        self.replay.push(s, a, r, s2, d)
        self.t_step += 1
        if (len(self.replay) < self.cfg.min_buffer_for_train
                or self.t_step % self.cfg.update_every != 0):
            return None
        return self.learn()

    # ----- core update --------------------------------------------------- #
    def learn(self) -> float:
        # 1. Sample a batch of experiences (s, a, r, s', done)
        states, actions, rewards, next_states, dones = self.replay.sample(self.cfg.batch_size)

        # 2. Compute Q targets (Double-DQN rule):
        #      a* selected by ONLINE net, value taken from TARGET net.
        with torch.no_grad():
            best_next_actions = self.qnetwork_local(next_states).argmax(dim=1, keepdim=True)
            Q_targets_next    = self.qnetwork_target(next_states).gather(1, best_next_actions).squeeze(1)
            Q_targets         = rewards + (self.cfg.gamma * Q_targets_next * (1.0 - dones))

        # 3. Expected Q from local net for the chosen actions
        Q_expected = self.qnetwork_local(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        # 4. Loss + backprop. Smooth-L1 (Huber) is robust to the +/-100 terminal
        #    rewards that would distort an MSE objective.
        loss = F.smooth_l1_loss(Q_expected, Q_targets)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.qnetwork_local.parameters(), self.cfg.grad_clip_norm)
        self.optimizer.step()

        # 5. Soft (Polyak) update of target net
        with torch.no_grad():
            for tp, lp in zip(self.qnetwork_target.parameters(),
                              self.qnetwork_local.parameters()):
                tp.data.mul_(1.0 - self.cfg.tau)
                tp.data.add_(self.cfg.tau * lp.data)

        return float(loss.item())

    def decay_epsilon(self):
        self.epsilon = max(self.cfg.epsilon_end, self.epsilon * self.cfg.epsilon_decay)

    def save(self, path: str):
        os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
        torch.save({'local': self.qnetwork_local.state_dict(),
                    'target': self.qnetwork_target.state_dict(),
                    'epsilon': self.epsilon}, path)

    def load(self, path: str):
        ckpt = torch.load(path, map_location=self.device, weights_only=False)
        self.qnetwork_local.load_state_dict(ckpt['local'])
        self.qnetwork_target.load_state_dict(ckpt['target'])
        self.epsilon = ckpt.get('epsilon', self.cfg.epsilon_end)

## 7. Training loop

Trains for `N_EPISODES` episodes, tracking episode rewards, the 100-episode rolling mean, and per-update loss. Saves the best snapshot by rolling-mean reward to `models/dddqn_best.pt`.

In [ ]:
def train(agent: DuelingDoubleDQNAgent,
          n_episodes: int = N_EPISODES,
          max_steps: int = MAX_STEPS,
          model_path: str = 'models/dddqn_best.pt',
          log_every: int = 20):
    env = make_env()
    rewards: List[float]   = []
    losses: List[float]    = []
    mean100: List[float]   = []
    window = deque(maxlen=100)
    best_mean = -float('inf'); solved_episode = None

    t0 = time.time()
    for ep in range(1, n_episodes + 1):
        state, _ = env.reset(seed=cfg.seed + ep)
        ep_reward = 0.0
        for _ in range(max_steps):
            action = agent.select_action(state, greedy=False)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            loss = agent.step(state, action, reward, next_state, done)
            if loss is not None: losses.append(loss)
            state = next_state; ep_reward += reward
            if done: break

        agent.decay_epsilon()
        rewards.append(ep_reward); window.append(ep_reward)
        m100 = float(np.mean(window)); mean100.append(m100)

        if len(window) >= 50 and m100 > best_mean:
            best_mean = m100
            agent.save(model_path)
        if solved_episode is None and len(window) == 100 and m100 >= 200.0:
            solved_episode = ep

        if ep % log_every == 0 or ep == 1:
            print(f'Ep {ep:4d}/{n_episodes} | R={ep_reward:7.2f} | mean100={m100:7.2f} '
                  f'| eps={agent.epsilon:.3f} | buf={len(agent.replay):6d} '
                  f"| loss={(losses[-1] if losses else float('nan')):.4f}")

    env.close()
    print(f"\nDone in {(time.time()-t0)/60:.1f} min. Best mean100={best_mean:.2f}. "
          f'Solved at: {solved_episode}.')
    return rewards, losses, mean100, solved_episode, best_mean

agent = DuelingDoubleDQNAgent(cfg)
rewards, losses, mean100, solved_episode, best_mean = train(agent)

## 8. Plots

Reward curve (per episode + 100-episode rolling mean) and loss curve (per gradient step).

In [ ]:
def plot_rewards(rewards, mean100, out_path='plots/rewards.png'):
    os.makedirs(os.path.dirname(out_path) or '.', exist_ok=True)
    x = np.arange(1, len(rewards) + 1)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(x, rewards, alpha=0.30, color='#1f77b4', label='Episode reward')
    ax.plot(x, mean100, color='#1f77b4', lw=2, label='100-episode mean')
    ax.axhline(200, ls='--', color='#2ca02c', lw=1.2, label='Solved threshold (=200)')
    ax.set_xlabel('Episode'); ax.set_ylabel('Total reward')
    ax.set_title('Training rewards on LunarLander')
    ax.grid(alpha=0.3); ax.legend(loc='lower right')
    fig.tight_layout(); fig.savefig(out_path, dpi=150); plt.show()
    return out_path

def plot_losses(losses, out_path='plots/loss.png'):
    os.makedirs(os.path.dirname(out_path) or '.', exist_ok=True)
    arr = np.asarray(losses)
    if len(arr) == 0:
        print('No losses to plot.'); return None
    ema = np.zeros_like(arr, dtype=np.float64); ema[0] = arr[0]
    for i in range(1, len(arr)):
        ema[i] = 0.01 * arr[i] + 0.99 * ema[i-1]
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(arr, alpha=0.20, color='#d62728', label='Per-step loss')
    ax.plot(ema, color='#d62728', lw=2.0, label='EMA-smoothed')
    ax.set_yscale('log')
    ax.set_xlabel('Gradient step'); ax.set_ylabel('Huber loss (log scale)')
    ax.set_title('Training loss vs. gradient step')
    ax.grid(alpha=0.3, which='both'); ax.legend(loc='upper right')
    fig.tight_layout(); fig.savefig(out_path, dpi=150); plt.show()
    return out_path

plot_rewards(rewards, mean100)
plot_losses(losses)

## 9. Greedy evaluation

Loads the best checkpoint and runs `N_EVAL` deterministic ($\varepsilon = 0$) roll-outs on unseen seeds. Reports mean, std, and the success rate (fraction of episodes scoring above 200).

In [ ]:
def evaluate(model_path: str = 'models/dddqn_best.pt', n_episodes: int = 30, seed: int = 1234):
    if not os.path.exists(model_path):
        print(f'No checkpoint at {model_path}; skipping eval.'); return None
    env = make_env()
    eval_agent = DuelingDoubleDQNAgent(cfg)
    eval_agent.load(model_path); eval_agent.qnetwork_local.eval()
    returns = []
    for ep in range(n_episodes):
        state, _ = env.reset(seed=seed + ep)
        ep_r = 0.0; done = False
        while not done:
            a = eval_agent.select_action(state, greedy=True)
            state, r, term, trunc, _ = env.step(a)
            ep_r += r; done = term or trunc
        returns.append(ep_r)
    env.close()
    returns = np.array(returns)
    print(f'Greedy evaluation over {n_episodes} episodes: '
          f'mean={returns.mean():.2f}, std={returns.std():.2f}')
    print(f'Success rate (>200): {(returns > 200).mean()*100:.0f}%')
    return returns

eval_returns = evaluate()

## 10. Demo GIF

Saves a short animation of the trained agent playing greedily, for the report. Frames are downscaled to keep file size and RAM manageable.

In [ ]:
def record_gif(model_path: str = 'models/dddqn_best.pt',
               out_path: str = 'videos/agent_demo.gif',
               n_episodes: int = 3, seed: int = 42, fps: int = 15):
    try:
        from PIL import Image
    except ImportError:
        print('Pillow not installed; pip install Pillow'); return None
    if not os.path.exists(model_path):
        print(f'No checkpoint at {model_path}'); return None
    os.makedirs(os.path.dirname(out_path) or '.', exist_ok=True)

    env = make_env(render_mode='rgb_array')
    eval_agent = DuelingDoubleDQNAgent(cfg)
    eval_agent.load(model_path); eval_agent.qnetwork_local.eval()

    frames = []
    for ep in range(n_episodes):
        state, _ = env.reset(seed=seed + ep)
        done = False; t = 0
        while not done:
            if t % 2 == 0:           # subsample to save memory
                f = env.render()
                f = np.array(Image.fromarray(f).resize((300, 200)))
                frames.append(f)
            a = eval_agent.select_action(state, greedy=True)
            state, _, term, trunc, _ = env.step(a)
            done = term or trunc; t += 1
        # Hold last frame between episodes
        for _ in range(5): frames.append(frames[-1])
    env.close()
    imageio.mimsave(out_path, frames, fps=fps, loop=0)
    print(f'Saved {out_path} ({len(frames)} frames, '
          f'{os.path.getsize(out_path)/1024:.1f} KB)')
    return out_path

record_gif()

---

**That's it.** Outputs are saved under `models/`, `plots/`, `videos/` next to the notebook.

To match the report's headline numbers exactly, set `N_EPISODES = 1000` at the top of section 3 and re-run from cell 6 (training onwards). The original 1000-episode run took 19.6 minutes on a single CPU and solved at episode 462 with a peak 100-episode mean reward of 252.87.